# SO-101 learning loop — cloud training leg (Colab / Kaggle T4)

**What this does.** Trains an **ACT** behaviour-cloning policy on our SO-101 datasets
(bus-log → LeRobot conversions, see `training/to_lerobot_dataset.py`) on a **free GPU**.
This is the *fix* step of the loop this repo demos — *run → capture → find the failure →
fix → run again* — moved to the cloud because superlocal has no NVIDIA GPU.

| stage | where | time (T4) |
|---|---|---|
| setup (clone + pip + unpack) | this notebook | ~5 min |
| ACT 6000 steps, state-only | this notebook | ~25–40 min |
| sanity eval + checkpoint zip | this notebook | ~2 min |
| **total** | | **~40–50 min** — SmolVLA with cameras: add 1–2 h |

**What to upload.** ONE file, built on superlocal by `scripts/pack_for_colab.sh`
(runbook: `training/colab/README.md`):

* `robot_learning_colab_bundle.zip` — repo essentials + `dataset.zip` + MANIFEST, **or**
* just `dataset.zip`, **or**
* nothing — if the dataset lives on the Hugging Face hub, skip uploads and point
  `--dataset.repo_id=<user>/<dataset>` in the train cell instead.

**Honesty note.** This notebook has **not** been executed on Colab — it can't be from
here. It was verified locally against lerobot **0.7.0** (the version `pip install
'lerobot[training]'` resolves today): JSON-linted, and the adapter → `lerobot-train` →
checkpoint-load → `select_action` path was smoke-run end-to-end on
`training/colab/_fixture` with the CPU training venv. Flag names in the train cell match
0.7: top-level `--steps` / `--batch_size` (older docs' `--training.steps` is gone), and
`--policy.push_to_hub=false` is required for local runs.


In [ ]:
# 1 · PLATFORM — Colab or Kaggle; everything below writes under BASE.
# (Local Jupyter is intentionally rejected: this notebook is the cloud leg.)
import os
from pathlib import Path

if os.path.exists("/content"):            # Google Colab
    BASE = Path("/content")
elif os.path.exists("/kaggle/working"):   # Kaggle (2×T4, 30 h/week — see README)
    BASE = Path("/kaggle/working")
else:
    raise RuntimeError("Open this notebook on Colab or Kaggle — not on superlocal.")
print("base dir:", BASE)


In [ ]:
# 2 · SETUP — clone the repo, unpack the dataset, install lerobot.
# GPU torch: on Colab/Kaggle the preinstalled CUDA torch is what we want; pip may
# rebuild torch to satisfy lerobot[training] — fine in the cloud. (superlocal uses a
# CPU-only training/.venv only because it has no NVIDIA GPU; that rule doesn't travel.)
import pathlib

REPO_URL = "https://github.com/qalarc/robot_learning.git"
REPO_DIR = BASE / "robot_learning"
if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

# dataset: either a bare dataset.zip, the full bundle (which contains one), or a
# Kaggle Dataset input. The zip holds a LeRobot-v3 dataset at repo_id local/so101.
DATA_ROOT = BASE / "datasets"
bundle = BASE / "robot_learning_colab_bundle.zip"
if (BASE / "dataset.zip").exists():
    src_zip = BASE / "dataset.zip"
elif bundle.exists():
    !unzip -o -q {bundle} -d {BASE}/bundle
    src_zip = BASE / "bundle" / "dataset.zip"
elif pathlib.Path("/kaggle/input").exists():
    hits = sorted(pathlib.Path("/kaggle/input").rglob("dataset.zip"))
    assert hits, "no dataset.zip under /kaggle/input — upload the bundle as a Kaggle Dataset (README step 2)"
    src_zip = hits[0]
else:
    raise AssertionError(
        "upload robot_learning_colab_bundle.zip (or dataset.zip) first — training/colab/README.md step 2"
    )
!unzip -o -q {src_zip} -d {DATA_ROOT}
!pip install -q 'lerobot[training]'

import lerobot
print("lerobot", lerobot.__version__, "| dataset unpacked to", DATA_ROOT)


In [ ]:
# 3 · INSPECT — what did we actually capture?
from lerobot.datasets.lerobot_dataset import LeRobotDataset

REPO_ID = "local/so101"          # pack_for_colab.sh always writes this repo_id
DS_DIR = DATA_ROOT / "so101"     # NB: lerobot 0.7 takes `root` as the dataset dir itself
ds = LeRobotDataset(REPO_ID, root=DS_DIR)
info = ds.meta.info

print("repo_id :", REPO_ID)
print("episodes:", ds.num_episodes, "| frames:", ds.num_frames, "| fps:", info.get("fps"))
print("robot   :", info.get("robot_type"))
try:
    print("tasks   :", sorted(ds.meta.tasks["task"].tolist()))
except Exception:
    pass
for k, v in sorted(ds.features.items()):
    if k in ("observation.state", "action") or k.startswith("observation.images.") or k.startswith("observation.env"):
        print(f"  {k:34s} {v['dtype']:8s} {v['shape']}")

HAS_IMAGES = any(k.startswith("observation.images.") for k in ds.features)
print("camera frames:", "yes — SmolVLA possible" if HAS_IMAGES else "no (state-only log — ACT only)")


In [ ]:
# 4 · ADAPTER — make the dataset ACT-trainable (a lerobot 0.7 quirk).
# ACT refuses state-only datasets: configuration_act.py raises unless the dataset has
# an `observation.images.*` feature OR an `observation.environment_state` feature.
# Bus logs carry proprioception only (7 floats: 6 joints + gripper), so we mirror
# observation.state into observation.environment_state — same numbers, extra feature
# name — and ACT trains on proprioception alone. The captured dataset is untouched;
# we write a sibling copy so re-packing never fights a mutated dataset.
import shutil

from lerobot.datasets.lerobot_dataset import LeRobotDataset

if HAS_IMAGES or "observation.environment_state" in ds.features:
    TRAIN_REPO_ID, TRAIN_ROOT = REPO_ID, DS_DIR
    print("dataset already ACT-ready — adapter skipped")
else:
    TRAIN_REPO_ID, TRAIN_ROOT = "local/so101_act", DATA_ROOT / "so101_act"
    if TRAIN_ROOT.exists():
        shutil.rmtree(TRAIN_ROOT)  # LeRobotDataset.create demands a fresh dir
    features = dict(ds.features)
    features["observation.environment_state"] = {
        "dtype": "float32",
        "shape": (7,),
        "names": {"joints": None},
    }
    dst = LeRobotDataset.create(
        repo_id=TRAIN_REPO_ID,
        fps=info.get("fps", 30),
        features=features,
        root=TRAIN_ROOT,
        robot_type=info.get("robot_type", "so101"),
        use_videos=False,
        image_writer_threads=0,
        image_writer_processes=0,
    )
    prev_ep = None
    for i in range(ds.num_frames):
        frame = ds[i]
        ep = int(frame["episode_index"])
        if prev_ep is not None and ep != prev_ep:
            dst.save_episode()
        dst.add_frame(
            {
                "observation.state": frame["observation.state"],
                "observation.environment_state": frame["observation.state"],
                "action": frame["action"],
                "task": frame["task"],
            }
        )
        prev_ep = ep
    dst.save_episode()
    dst.finalize()
    print("ACT-ready copy:", TRAIN_ROOT, f"({ds.num_frames} frames)")


In [ ]:
# 5 · FLAGS — lerobot's CLI moves between releases; print the truth for the version
# pip just installed. If a flag in cell 6 ever errors, look its new name up here
# (verified against 0.7.0: --steps / --batch_size / --save_freq are top-level).
!lerobot-train --help | head -40
!lerobot-train --help | grep -E "steps|batch_size|save_freq|push_to_hub|eval_split"


In [ ]:
# 6 · TRAIN — ACT behaviour cloning on the free T4.
# Defaults sized for the Colab bubble: 6000 steps, batch 8, checkpoint every 2000 →
# ~25–40 min on a T4 for a state-only SO-101 dataset; the whole notebook stays well
# under 2 h. Scaling: 2000 steps = pitch-demo quality, 12000 = best-effort.
# 0.7 gotchas verified locally: (a) --policy.push_to_hub=false is REQUIRED for local
# runs (else validate() demands a hub repo_id); (b) an existing --output_dir fails
# unless --resume, so we clear it first — safe to Run-all twice.
import shutil

POLICY = "act"   # 'smolvla' only works with camera frames — see the note below
STEPS = 6000
BATCH = 8

OUT_DIR = BASE / "runs" / "act_so101"
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

!lerobot-train \
  --dataset.repo_id={TRAIN_REPO_ID} \
  --dataset.root={TRAIN_ROOT} \
  --policy.type={POLICY} \
  --policy.push_to_hub=false \
  --steps={STEPS} --batch_size={BATCH} \
  --save_freq=2000 --log_freq=200 --num_workers=2 \
  --seed=7 --job_name=so101_colab --output_dir={OUT_DIR}

# SmolVLA variant — ONLY when the dataset has camera frames (state-only logs cannot
# train it: modeling_smolvla.py raises "SmolVLA requires at least one camera").
# Cell 3 prints whether your dataset qualifies. On a T4 keep it modest:
#   --policy.type=smolvla --steps=4000 --batch_size=4   (~2 h with images)
# Image-bearing datasets: either pack web-recorder episode files with frames, or use
# any LeRobot dataset on the HF hub: --dataset.repo_id=<user>/<ds> and drop --dataset.root.
# Optional honest eval during training (needs >=2 episodes per task, else it starves):
#   append --dataset.eval_split=0.25


In [ ]:
# 7 · SANITY EVAL + the nines table.
# Env rollouts: no gym env for our SO-101 sim ships inside lerobot, so there is
# nothing to roll out in the cloud — SKIP (this is the else branch of the spec).
# Rollouts happen back on superlocal (sim/armsmith_web + the stall loop). Here we do
# the cloud-side sanity check instead: load the last checkpoint and score its
# predictions against the final episode's recorded actions.
import torch

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import get_policy_class

run_dir = BASE / "runs" / "act_so101"
ckpts = sorted((run_dir / "checkpoints").glob("0*"))
assert ckpts, "no checkpoints — did cell 6 train?"
ckpt = ckpts[-1] / "pretrained_model"
print("checkpoint:", ckpt)

policy = get_policy_class(POLICY).from_pretrained(str(ckpt))
policy.eval()
policy.reset()

train_ds = LeRobotDataset(TRAIN_REPO_ID, root=TRAIN_ROOT)
hold = min(20, train_ds.num_frames // 4)  # tail slice = the last episode's ending
errs, base = [], []
with torch.no_grad():
    for i in range(train_ds.num_frames - hold, train_ds.num_frames):
        f = train_ds[i]
        obs = {"observation.state": f["observation.state"].unsqueeze(0), "task": [f["task"]]}
        if "observation.environment_state" in train_ds.features:
            obs["observation.environment_state"] = f["observation.environment_state"].unsqueeze(0)
        pred = policy.select_action(obs)          # normalization handled internally
        tgt = f["action"].unsqueeze(0)
        errs.append((pred - tgt).abs().mean().item())
        base.append(tgt.abs().mean().item())
l1, b1 = sum(errs) / len(errs), sum(base) / len(base)
print(f"held-out action L1: {l1:.4f}   (zero-action baseline {b1:.4f})")
print("smoke check only — real numbers come from rolling this checkpoint in the sim on superlocal.")
print()
print(
    "nines summary (fill from superlocal runs/metrics.json, before → after):\n"
    "  episodes / successes / stalls / escalations / mean_time_to_recover_s\n"
    "  escalation_rate  0.0 → ?\n"
    "  ep_success_rate  0.0 → ?"
)


In [ ]:
# 8 · EXPORT — zip every checkpoint + training config for download, then pull it.
import shutil

out_zip = BASE / "act_so101_checkpoints.zip"
if out_zip.exists():
    out_zip.unlink()  # pathlib unlink of our own artifact — no shell rm needed
shutil.make_archive(str(out_zip.with_suffix("")), "zip", str(BASE / "runs" / "act_so101"))
print("checkpoints zip:", out_zip, f"({out_zip.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files  # Colab only; Kaggle shows it in the right panel

    files.download(str(out_zip))
except Exception as exc:
    print(f"download it from the file browser instead ({type(exc).__name__})")
print("next: unpack on superlocal and roll the checkpoint in sim/armsmith_web — the loop closes there.")
